#### Loading Pre-trained Glove vectors (Global vectors for word representation)

Is a combination of two approaches : 

1) Count based (co-occurance matrix)
2) Predictive methods (word2vec)

Key insight : Ratio of co-occurence probabilities encode meaning

P(ik) = X(ik)/X(i)

X(ik) : number of times word k occured in context of i 
X(i) : total number of words in context of i 

Ration of Probabilities : P(ik)/P(jk)

Example: “ice” vs “steam” in relation to “solid”: ratio is high

Example: “ice” vs “steam” in relation to “gas”: ratio is low

In [1]:
# lets implement this Glove
import numpy as np 
import torch 

In [7]:
# make a corpus for training 
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat chased the dog",
    "the dog chased the cat"
]

In [8]:
# lets tokenize each sentence 
from nltk.tokenize import word_tokenize
tokens = [word_tokenize(sentence) for sentence in corpus]

for sent_tokens in tokens:
    print(sent_tokens)

['the', 'cat', 'sat', 'on', 'the', 'mat']
['the', 'dog', 'sat', 'on', 'the', 'log']
['the', 'cat', 'chased', 'the', 'dog']
['the', 'dog', 'chased', 'the', 'cat']


In [9]:
# lets make a vocabulary 
word2idx = {}
idx2word = {}

# set for unique words
unique_words = set()
idx = 0
for sent in tokens:
    for word in sent:
        if word not in unique_words:
            word2idx[word] = idx 
            idx2word[idx] = word 
            idx += 1 
            unique_words.add(word)

# length of vocabulary 
print(f"Length of vocabulary : {len(unique_words)}")

Length of vocabulary : 8


Make a co-occurance matrix that keeps count of how many times a word occured in context of other word

In [10]:
factor_matrix = np.zeros((len(word2idx), len(word2idx)))
print(f"Shape of a co-occurence matrix : {factor_matrix.shape}")

Shape of a co-occurence matrix : (8, 8)


In [12]:
# how many times a word appeared in context of another word (contex_size=2)
window_size = 2 

for sent in tokens:
    for i, word in enumerate(sent):
        # I have a word and want to calculate the co-occurance 
        left = max(i - window_size, 0)
        right = min(len(sent), i + window_size + 1)
        for j in range(left, right):
            if i != j:
                factor_matrix[word2idx[word]][word2idx[sent[j]]] += 1

print(factor_matrix)


[[0. 4. 4. 2. 1. 4. 1. 4.]
 [4. 0. 1. 1. 0. 0. 0. 2.]
 [4. 1. 0. 2. 0. 1. 0. 0.]
 [2. 1. 2. 0. 1. 1. 1. 0.]
 [1. 0. 0. 1. 0. 0. 0. 0.]
 [4. 0. 1. 1. 0. 0. 0. 2.]
 [1. 0. 0. 1. 0. 0. 0. 0.]
 [4. 2. 0. 0. 0. 2. 0. 0.]]


## Glove Model

In [91]:
# lets make the model 
import torch.nn as nn 

class Glove(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(Glove, self).__init__()
        self.wi = nn.Embedding(vocab_size, embedding_dim)
        self.wj = nn.Embedding(vocab_size, embedding_dim)
        self.bi = nn.Embedding(vocab_size, 1)
        self.bj = nn.Embedding(vocab_size, 1)

        # init
        nn.init.xavier_uniform_(self.wi.weight)
        nn.init.xavier_uniform_(self.wj.weight)
        nn.init.zeros_(self.bi.weight)
        nn.init.zeros_(self.bj.weight)

    def forward(self, i, j):
        dot = (self.wi(i) * self.wj(j)).sum(1)
        return dot + self.bi(i).squeeze() + self.bj(j).squeeze()

In [92]:
embedding_dim = 10
model = Glove(len(word2idx), embedding_dim)
print(model)

Glove(
  (wi): Embedding(8, 10)
  (wj): Embedding(8, 10)
  (bi): Embedding(8, 1)
  (bj): Embedding(8, 1)
)


In [93]:
# hyperparameters 
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
# weighting function f(x)
def weighting_fn(x, x_max=100, alpha=0.75):
    return (x / x_max) ** alpha if x < x_max else 1

# Loss function
def glove_loss(pred, x_ij, weight):
    return torch.mean(weight * (pred - torch.log(x_ij))**2)

In [94]:
# training loop

pairs = []
weights = []
values = []

for i in range(len(word2idx)):
    for j in range(len(word2idx)):
        if factor_matrix[i][j] > 0:
            pairs.append((i, j))
            weights.append(weighting_fn(factor_matrix[i][j]))
            values.append(factor_matrix[i][j])

pairs = torch.tensor(pairs)
weights = torch.tensor(weights, dtype=torch.float32)
values = torch.tensor(values, dtype=torch.float32)

epochs = 200
for epoch in range(epochs):
    i = pairs[:,0]
    j = pairs[:,1]

    preds = model(i, j)
    loss = glove_loss(preds, values, weights)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch+1) % 50 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

Epoch 50/200, Loss: 0.0003
Epoch 100/200, Loss: 0.0000
Epoch 150/200, Loss: 0.0000
Epoch 200/200, Loss: 0.0000


In [95]:
embeddings = model.wi.weight.data + model.wj.weight.data
for word, idx in word2idx.items():
    print(f"{word}: {embeddings[idx][:5]}")  # show first 5 dims

the: tensor([ 0.9451,  0.4915,  0.3526, -0.0752, -0.0124])
cat: tensor([ 0.1042, -0.0762,  0.1883, -0.5085,  0.0032])
sat: tensor([ 0.8093,  1.0063,  0.0559, -0.5631,  0.1744])
on: tensor([ 0.1118,  1.1821,  0.5117, -0.2265,  0.2486])
mat: tensor([-0.0643,  0.5655, -0.3914, -0.0630, -0.4447])
dog: tensor([ 1.2245, -0.0209,  0.6465,  0.5961, -0.1395])
log: tensor([ 0.4717, -0.0480, -0.3363,  1.4233, -0.6351])
chased: tensor([ 0.1177,  0.0072, -0.1440, -0.2210,  0.7123])
